In [14]:
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt

from src.geometry import RectangleSpace
from src.fields import InternalScalarField, SimpleWallBC, insulated_rod_ends
from src.cv_method import Diffusion

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [15]:
# Create Rod and cut to 10 pieces
rod = RectangleSpace([0],[1000]).grid([10],'cell')
boundary_temp_bcs = insulated_rod_ends(100,500)
# Create field object
rod_temp = InternalScalarField(grid=rod,bcs=boundary_temp_bcs)

# Create to-be-solvede object
rod_problem = Diffusion(rod_temp, diffusion_coefficient= 7)


In [17]:
# Assemble
matrix, vector = rod_problem.assemble()

print(matrix)
print(vector)


[[ 0.21 -0.07  0.    0.    0.    0.    0.    0.    0.    0.  ]
 [-0.07  0.14 -0.07  0.    0.    0.    0.    0.    0.    0.  ]
 [ 0.   -0.07  0.14 -0.07  0.    0.    0.    0.    0.    0.  ]
 [ 0.    0.   -0.07  0.14 -0.07  0.    0.    0.    0.    0.  ]
 [ 0.    0.    0.   -0.07  0.14 -0.07  0.    0.    0.    0.  ]
 [ 0.    0.    0.    0.   -0.07  0.14 -0.07  0.    0.    0.  ]
 [ 0.    0.    0.    0.    0.   -0.07  0.14 -0.07  0.    0.  ]
 [ 0.    0.    0.    0.    0.    0.   -0.07  0.14 -0.07  0.  ]
 [ 0.    0.    0.    0.    0.    0.    0.   -0.07  0.14 -0.07]
 [ 0.    0.    0.    0.    0.    0.    0.    0.   -0.07  0.21]]
[14.  0.  0.  0.  0.  0.  0.  0.  0. 70.]


In [ ]:
diag = np.diag(matrix)
upper_diag = np.diag(matrix,1)
lower_diag = np.diag(matrix,-1)
right_vec = vector
P_0 = upper_diag[0]/diag[0]
Q_0 = right_vec[0]/diag[0]
P_array = np.array([P_0])
Q_array = np.array([Q_0])
denom = np.array([diag[i]-lower_diag[i]*P_array[i-1] for i in range(9)])
P_array = np.concatenate([P_array,upper_diag/denom])
Q_array = np.concatenate([Q_array, [right_vec[i]*Q_array[i-1]/denom[i] for i in range(9)]])

T_array = np.array(Q[-2])
T_array = np.concatenate([[P_array[i]*T_array[i+1]+Q_array[i] for i in range(8)],T_array])
